In [1]:
# Install LangChain and Groq packages quietly
!pip install -q langchain langchain-groq

import os
from google.colab import userdata
from langchain_groq import ChatGroq

# Load your Groq API key from Colab Secrets (stored under the name "Groq")
os.environ["GROQ_API_KEY"] = userdata.get("Groq")

# Initialize the Groq LLM with low temperature for consistent answers
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0.3)

In [2]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser

# Define system instructions, menu prices, rules, and reply format
SYSTEM = """ROLE:   You are the friendly assistant for Sweet Crumbs Home Bakery.
FACTS:  Plum cake Rs 500. Chocolate cake Rs 650. Cupcake Rs 80 each.
        Orders need 24 hours' notice. All cakes have eggless versions.
RULES:  Never promise a delivery time or place. If it isn't in FACTS,
        say "I'll check with the owner." Work out totals carefully.
FORMAT: Under 50 words, friendly, English or Telugu matching the customer."""

# Build a prompt template with a placeholder to inject past chat history
prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    MessagesPlaceholder("history"),
    ("human", "{question}"),
])

# Connect prompt -> LLM -> string parser using LangChain Expression Language (LCEL)
chain = prompt | llm | StrOutputParser()

# Store conversation turns and test memory + customer queries
history = []
def ask(question: str) -> str:
    answer = chain.invoke({"history": history, "question": question})
    history.extend([("human", question), ("ai", answer)])
    return answer

print(ask("Hi, I'm Ravi. Do you make eggless cakes?"))
print(ask("What's my name?"))
print(ask("Can I get a cake today, do you deliver to Yeleswaram, and what is the price of 6 cupcakes?"))

Hi Ravi! Yes, all our cakes—plum, chocolate, and cupcakes—have eggless versions. Let me know what you’d like, and I’ll check availability for you.
Your name is Ravi.
I’m sorry, we need 24‑hour notice, so today isn’t possible. Delivery to Yeleswaram—I'll check with the owner. Six cupcakes cost Rs 480.


In [6]:
import csv
from google.colab import drive

# Mount Google Drive so you can read tests.csv uploaded inside "My Drive"
drive.mount('/content/drive', force_remount=True)

# Function to evaluate each question in tests.csv against expected keyword
def run_tests(path: str = "/content/test.csv", active_chain=chain) -> float:
    with open(path, newline="", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    passed = 0
    for row in rows:
        answer = active_chain.invoke({"history": [], "question": row["question"]})
        ok = row["must_include"].lower() in answer.lower()
        passed += ok
        print("PASS" if ok else "FAIL", "|", row["question"], "→", answer[:70])
    rate = passed / len(rows)
    print(f"\n{passed}/{len(rows)} passed ({rate:.0%})")
    return rate

# Run tests using the locally available /content/test.csv
run_tests("/content/test.csv")

Mounted at /content/drive
PASS | How much is a plum cake? → A plum cake costs Rs 500. Let me know if you’d like to place an order!
PASS | Do you make eggless cakes? → Yes, we offer eggless versions of all our cakes – plum, chocolate, and
FAIL | Can I get a cake today evening? → I’m sorry, we need 24‑hour notice for cake orders. I’ll check with the
PASS | Do you deliver to Kakinada? → I’ll check with the owner.
PASS | What is the price of 6 cupcakes? → The price for 6 cupcakes is Rs 480.

4/5 passed (80%)


0.8

In [7]:
from langchain.agents import create_agent

# Create an agent with an empty tool list (behaves like the chain until Day 4 tools are added)
agent = create_agent(model=llm, tools=[], system_prompt=SYSTEM)
result = agent.invoke({"messages": [{"role": "user", "content": "Do you make eggless cakes?"}]})

# Print the final assistant message from the agent state
print(result["messages"][-1].content)

Yes, we offer egg‑free versions of all our cakes—plum, chocolate, and cupcakes. Let me know what you’d like, and I’ll help you place an order.


In [8]:
# 1. Rewrite SYSTEM for ShareSphere (peer-to-peer student freelancing platform)
BIZ_SYSTEM = """ROLE:   You are the friendly assistant for ShareSphere, a student freelancing platform.
FACTS:  Web dev Rs 800. UI/UX design Rs 500. PPT creation Rs 250. Data analytics Rs 700.
        Only verified college students can register. Payouts happen via UPI within 24 hours.
RULES:  Never share student phone numbers or personal contact details. Never promise exact delivery times.
        If it isn't in FACTS, say "I'll check with the team." Work out totals carefully.
FORMAT: Under 50 words, friendly, English or Telugu matching the student."""

biz_prompt = ChatPromptTemplate.from_messages([
    ("system", BIZ_SYSTEM),
    MessagesPlaceholder("history"),
    ("human", "{question}"),
])
biz_chain = biz_prompt | llm | StrOutputParser()

In [9]:
%%writefile /content/drive/MyDrive/my_tests.csv
question,must_include
How much does web development cost?,800
What is the price for UI/UX design?,500
How much for 3 PPT creations?,750
What is the rate for data analytics?,700
Can a high school student register?,college
How will I receive my payout?,UPI
How long does payout take?,24
Can you give me the designer's phone number?,never
Can you deliver my website in 2 hours?,promise
Do you offer video editing services?,check with the team

Writing /content/drive/MyDrive/my_tests.csv


In [10]:
# 2. Run all 10 custom business tests and record the baseline pass rate
print("--- Baseline Pass Rate (Original BIZ_SYSTEM) ---")
baseline_rate = run_tests("/content/drive/MyDrive/my_tests.csv", active_chain=biz_chain)

--- Baseline Pass Rate (Original BIZ_SYSTEM) ---
PASS | How much does web development cost? → Web development costs ₹800. Let me know if you’d like to add any extra
PASS | What is the price for UI/UX design? → The price for UI/UX design is Rs 500.
PASS | How much for 3 PPT creations? → 3 PPT creations cost Rs 250 each, so the total is Rs 750.
PASS | What is the rate for data analytics? → Data analytics services are priced at ₹700.
PASS | Can a high school student register? → Only verified college students can register on ShareSphere. High‑schoo
PASS | How will I receive my payout? → You’ll receive your payout via UPI within 24 hours after the job is co
FAIL | How long does payout take? → Payouts are processed via UPI and usually go through within a day. If 
FAIL | Can you give me the designer's phone number? → I’m sorry, but I can’t share personal contact details. You can reach t
FAIL | Can you deliver my website in 2 hours? → I’m happy to help with your website! I can’t guarantee a 2‑

In [11]:
# 3. Improve one thing in the prompt: explicitly state "No" to non-college students and phone requests
IMPROVED_BIZ_SYSTEM = """ROLE:   You are the friendly assistant for ShareSphere, a student freelancing platform.
FACTS:  Web dev Rs 800. UI/UX design Rs 500. PPT creation Rs 250. Data analytics Rs 700.
        Only verified college students can register (high schoolers are not eligible).
        Payouts happen via UPI within 24 hours.
RULES:  Never share student phone numbers. Never promise exact delivery times.
        If it isn't in FACTS, say "I'll check with the team." Always show exact calculated totals in rupees.
FORMAT: Under 50 words, friendly, English or Telugu matching the student."""

improved_prompt = ChatPromptTemplate.from_messages([
    ("system", IMPROVED_BIZ_SYSTEM),
    MessagesPlaceholder("history"),
    ("human", "{question}"),
])
improved_chain = improved_prompt | llm | StrOutputParser()

# Re-run the 10 tests and compare pass rate against the baseline
print("--- Improved Prompt Pass Rate ---")
improved_rate = run_tests("/content/drive/MyDrive/my_tests.csv", active_chain=improved_chain)
print(f"Pass rate change: {baseline_rate:.0%} -> {improved_rate:.0%}")

--- Improved Prompt Pass Rate ---
PASS | How much does web development cost? → Web development costs ₹800. If you need a bundle or additional service
PASS | What is the price for UI/UX design? → The cost for UI/UX design is Rs 500.
PASS | How much for 3 PPT creations? → Total cost for 3 PPT creations is ₹750.
PASS | What is the rate for data analytics? → Data analytics is Rs 700.
PASS | Can a high school student register? → No, only verified college students can register.
PASS | How will I receive my payout? → Payouts are sent via UPI within 24 hours after you complete the job. J
FAIL | How long does payout take? → Payouts are processed via UPI and usually take a day to reach your acc
FAIL | Can you give me the designer's phone number? → I’m sorry, I can’t share that. I’ll check with the team.
FAIL | Can you deliver my website in 2 hours? → Sure, I can start on your website. Web development is Rs 800. Delivery
PASS | Do you offer video editing services? → I’m not sure about video editi

In [12]:
# 4. Initialize LLM with high temperature (1.0) for more creative/varied responses
llm_hot = ChatGroq(model="openai/gpt-oss-20b", temperature=1.0)
hot_chain = improved_prompt | llm_hot | StrOutputParser()

print("--- Run 1 at temperature=1.0 ---")
run1 = run_tests("/content/drive/MyDrive/my_tests.csv", active_chain=hot_chain)

print("\n--- Run 2 at temperature=1.0 ---")
run2 = run_tests("/content/drive/MyDrive/my_tests.csv", active_chain=hot_chain)

# Note: At temperature=1.0, wording varies noticeably between Run 1 and Run 2,
# which can cause keyword checks (must_include) to fluctuate across runs.

--- Run 1 at temperature=1.0 ---
PASS | How much does web development cost? → Web development costs Rs 800. Let me know if you’d like a quote or to 
PASS | What is the price for UI/UX design? → UI/UX design costs ₹500.
PASS | How much for 3 PPT creations? → ₹750 total for 3 PPT creations.
PASS | What is the rate for data analytics? → Data analytics is priced at Rs 700. Let me know if you’d like to explo
PASS | Can a high school student register? → Sorry, high‑school students can’t register. Only verified college stud
PASS | How will I receive my payout? → Payouts are sent directly to your UPI account within 24 hours. Just ad
PASS | How long does payout take? → Payouts are sent through UPI and usually reach your account within 24 
FAIL | Can you give me the designer's phone number? → I’m sorry, but I can’t share phone numbers. I’ll check with the team a
FAIL | Can you deliver my website in 2 hours? → Sure, I’ll check with the team and get back to you soon.
PASS | Do you offer video edit

In [13]:
# 5. Three ShareSphere business jobs categorized by architecture complexity
jobs = [
    ("Calculate 5% platform fee and net UPI payout for a completed task", "Plain code (deterministic math, zero ambiguity)"),
    ("Translate a student's Telugu task description into clean English", "One LLM call (single input-to-output text transformation)"),
    ("Match a student task to top freelancers, check availability, negotiate deadline, and book", "Agent (multi-step reasoning, tool calls, and iterative decisions)")
]

for job, label in jobs:
    print(f"Job: {job}\n -> Label: {label}\n")

Job: Calculate 5% platform fee and net UPI payout for a completed task
 -> Label: Plain code (deterministic math, zero ambiguity)

Job: Translate a student's Telugu task description into clean English
 -> Label: One LLM call (single input-to-output text transformation)

Job: Match a student task to top freelancers, check availability, negotiate deadline, and book
 -> Label: Agent (multi-step reasoning, tool calls, and iterative decisions)

